In [1]:
import pandas as pd

In [11]:
df1=pd.read_csv("Bitext_Sample_Customer_Service_Training_Dataset.csv")
df2=pd.read_csv("Bitext_Sample_Customer_Service_Testing_Dataset.csv")
df3=pd.read_csv("Bitext_Sample_Customer_Service_Validation_Dataset.csv")


In [12]:
df=pd.concat([df1,df2,df3])

In [14]:
df.to_csv('combined.csv',index=False)

In [16]:
df=pd.read_csv('combined.csv')

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8175 entries, 0 to 8174
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   utterance  8175 non-null   object
 1   intent     8175 non-null   object
 2   category   8175 non-null   object
 3   tags       8175 non-null   object
dtypes: object(4)
memory usage: 255.6+ KB


In [18]:
df.shape

(8175, 4)

In [20]:
df.head()

,utterance,intent,category,tags
0,would it be possible to cancel the order I made?,cancel_order,ORDER,BIP
1,cancelling order,cancel_order,ORDER,BK
2,I need assistance canceling the last order I h...,cancel_order,ORDER,B
3,problem with canceling the order I made,cancel_order,ORDER,B
4,I don't know how to cancel the order I made,cancel_order,ORDER,B


In [19]:
print("Number of intents:", df["intent"].nunique())
print(df["intent"].value_counts())

Number of intents: 27
intent
get_invoice                 324
check_invoice               324
payment_issue               323
review                      315
track_refund                308
set_up_shipping_address     307
place_order                 306
track_order                 305
cancel_order                305
change_order                304
delivery_options            302
check_refund_policy         302
delivery_period             301
contact_customer_service    299
create_account              298
check_cancellation_fee      298
recover_password            298
complaint                   298
delete_account              298
check_payment_methods       297
change_shipping_address     297
contact_human_agent         297
registration_problems       296
newsletter_subscription     295
get_refund                  294
edit_account                294
switch_account              290
Name: count, dtype: int64


In [29]:

df = df[["utterance", "intent"]].copy()

df["utterance"] = df["utterance"].astype(str).str.strip()
df["intent"] = df["intent"].astype(str).str.strip()

print(df.shape)
print(df.isnull().sum())

(8175, 2)
utterance    0
intent       0
dtype: int64


In [22]:
from sklearn.model_selection import train_test_split

X = df["utterance"]
y = df["intent"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 6540
Testing samples: 1635


In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    max_features=20000
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training matrix:", X_train_tfidf.shape)
print("Testing matrix:", X_test_tfidf.shape)

Training matrix: (6540, 3041)
Testing matrix: (1635, 3041)


In [31]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train_tfidf, y_train)

print("Model training completed!")

Model training completed!


In [32]:
from sklearn.metrics import accuracy_score,classification_report

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))


Accuracy: 0.9914373088685016

Classification Report:
                          precision    recall  f1-score   support

            cancel_order       0.95      1.00      0.98        61
            change_order       1.00      0.97      0.98        61
 change_shipping_address       0.98      0.97      0.97        59
  check_cancellation_fee       1.00      1.00      1.00        60
           check_invoice       1.00      1.00      1.00        65
   check_payment_methods       1.00      0.98      0.99        59
     check_refund_policy       0.98      1.00      0.99        60
               complaint       1.00      1.00      1.00        60
contact_customer_service       1.00      1.00      1.00        60
     contact_human_agent       1.00      1.00      1.00        59
          create_account       0.98      0.95      0.97        60
          delete_account       1.00      0.98      0.99        60
        delivery_options       0.98      1.00      0.99        60
         delivery_peri

In [34]:
new_queries = [
    "I want to cancel my order",
    "I need help with my payment"
]

new_queries_tfidf = tfidf.transform(new_queries)
predictions = model.predict(new_queries_tfidf)

for query, intent in zip(new_queries, predictions):
    print("User:", query)
    print("Predicted intent:", intent)
    print()

User: I want to cancel my order
Predicted intent: cancel_order

User: I need help with my payment
Predicted intent: payment_issue



In [35]:
import joblib

joblib.dump(model, "intent_classifier.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

print("Model and vectorizer saved!")


Model and vectorizer saved!


In [36]:
import streamlit as st
import joblib

# ----------------------------
# Page configuration
# ----------------------------
st.set_page_config(
    page_title="Intent Classifier",
    page_icon="💬",
    layout="centered"
)

# ----------------------------
# Load the saved model
# ----------------------------
@st.cache_resource
def load_artifacts():
    model = joblib.load("intent_classifier.pkl")
    tfidf = joblib.load("tfidf_vectorizer.pkl")
    return model, tfidf

try:
    model, tfidf = load_artifacts()
except FileNotFoundError:
    st.error(
        "Model files not found. Save intent_classifier.pkl and "
        "tfidf_vectorizer.pkl in the same folder as app.py."
    )
    st.stop()

# ----------------------------
# UI
# ----------------------------
st.title("💬 Intent Classification")
st.write(
    "Enter a sentence below and the model will predict the user's intent."
)

st.divider()

utterance = st.text_area(
    "Your message",
    placeholder="Example: I want to check my order status",
    height=120
)

if st.button("Predict Intent", type="primary", use_container_width=True):
    if not utterance.strip():
        st.warning("Please enter a message first.")
    else:
        features = tfidf.transform([utterance.strip()])
        prediction = model.predict(features)[0]

        st.subheader("Prediction")
        st.success(f"Predicted intent: **{prediction}**")

        # Show confidence when the classifier supports predict_proba.
        if hasattr(model, "predict_proba"):
            probabilities = model.predict_proba(features)[0]
            confidence = float(probabilities.max())
            st.metric("Model confidence", f"{confidence:.1%}")

            # Optional: show the most likely alternative intents.
            classes = model.classes_
            top_indices = probabilities.argsort()[::-1][:3]

            st.write("Top matching intents")
            for idx in top_indices:
                st.write(f"- **{classes[idx]}** — {probabilities[idx]:.1%}")
        else:
            st.info(
                "This model does not provide probability estimates, "
                "so a confidence score is unavailable."
            )

st.divider()
st.caption("Prototype • TF-IDF + Scikit-learn intent classifier")


2026-10-10 21:38:52.578 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-10 21:38:52.582 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-10 21:38:53.774 
  command:

    streamlit run C:\Users\SRIHITH REDDY\AppData\Roaming\Python\Python313\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-10-10 21:38:53.775 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-10 21:38:53.777 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-10 21:38:53.778 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-10 21:38:53.866 Thread 'MainThread': missing ScriptRunContext! This warni

DeltaGenerator()